In [1]:
# Import the libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### Load the data sets

Below are the datasets we have pulled for predicting the calfire acres burnt in california.
The datasets are below as follows, 
 1. Calfire
 2. Weather
 3. Vegetation
 4. Topography
 5. Drought
 6. Spatial

In [2]:
# Load the datasets

df_fire = pd.read_csv('../data/interim/calfire.csv')
df_weather = pd.read_csv('../data/interim/calfire_weather.csv')
df_vegetation = pd.read_csv('../data/interim/calfire_vegetation.csv')
df_topography = pd.read_csv('../data/interim/calfire_topography.csv')
df_drought = pd.read_csv('../data/interim/calfire_drought.csv')
df_spatial = pd.read_csv('../data/interim/calfire_spatial_data.csv')

### Shape of the datasets

In [3]:
# Functions

def print_shape(name, datasets):
    """
    This function is used to print the shape of the datasets
    Arguments:
        name: Name of the datasets in list
        datasets: List of the datasets
    Return:
        No return, just print the shape of the datasets
    """
    for dsname,ds in zip(name,datasets):
        print(f"Shape of {dsname} is {ds.shape} \n")

In [4]:
ds_names = ['calfire','weather','vegetation','topography','drought','spatial']
ds = [df_fire,df_weather,df_vegetation,df_topography,df_drought,df_spatial]

# Print the dataset shapes
print_shape(ds_names,ds)

Shape of calfire is (3509, 26) 

Shape of weather is (3509, 96) 

Shape of vegetation is (2655, 41) 

Shape of topography is (3405, 30) 

Shape of drought is (3499, 23) 

Shape of spatial is (3492, 11) 



### All columns

In [5]:
df_columns = pd.concat([pd.DataFrame({
    "Dataset": "fire",
    "Column": df_fire.columns
}),
pd.DataFrame({
    "Dataset": "weather",
    "Column": df_weather.columns
}),
pd.DataFrame({
    "Dataset": "vegetation",
    "Column": df_vegetation.columns
}),
pd.DataFrame({
    "Dataset": "topography",
    "Column": df_topography.columns
}),
pd.DataFrame({
    "Dataset": "drought",
    "Column": df_drought.columns
}),
pd.DataFrame({
    "Dataset": "spatial",
    "Column": df_spatial.columns
})])

### Feature selection 

### Fire Dataset

In [6]:
df_fire.columns

Index(['incident_name', 'incident_is_final', 'incident_date_last_update',
       'incident_date_created', 'incident_administrative_unit',
       'incident_administrative_unit_url', 'incident_county',
       'incident_location', 'incident_acres_burned', 'incident_containment',
       'incident_control', 'incident_cooperating_agencies',
       'incident_longitude', 'incident_latitude', 'incident_type',
       'incident_id', 'incident_url', 'incident_date_extinguished',
       'incident_dateonly_extinguished', 'incident_dateonly_created',
       'is_active', 'calfire_incident', 'notification_desired', 'end_date',
       'diff_days', 'bin_fire_days'],
      dtype='object')

In [7]:
fire_columns = ['incident_id','incident_is_final','incident_county','incident_acres_burned',
               'is_active']

# 'incident_name','incident_longitude','incident_latitude','incident_dateonly_created','end_date'

In [8]:
# Pick only the selected columns in the fire dataset
df_fire_selected = df_fire[fire_columns].copy()

# Filter out only active Incidents
df_fire_selected = df_fire_selected[df_fire_selected.is_active=='N']

# Filter out only for the Incidents that received all the data and finalized
df_fire_selected = df_fire_selected[df_fire_selected.incident_is_final=='Y']

# Remove the records that do not have the Incident acres burned, as it is the target variable for us.
# Any record which do not have the values for target variable is not useful for prediction
df_fire_selected = df_fire_selected[~df_fire_selected.incident_acres_burned.isna()]


df_fire_selected.drop(columns='is_active',inplace=True)
df_fire_selected.drop(columns='incident_is_final',inplace=True)

In [9]:
df_fire_selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3439 entries, 0 to 3507
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   incident_id            3439 non-null   object 
 1   incident_county        3429 non-null   object 
 2   incident_acres_burned  3439 non-null   float64
dtypes: float64(1), object(2)
memory usage: 107.5+ KB


In [10]:
df_fire_selected.columns

Index(['incident_id', 'incident_county', 'incident_acres_burned'], dtype='object')

### Weather Dataset

In [11]:
df_weather.columns

Index(['Unnamed: 0', 'incident_name', 'incident_is_final',
       'incident_date_last_update', 'incident_date_created',
       'incident_administrative_unit', 'incident_administrative_unit_url',
       'incident_county', 'incident_location', 'incident_acres_burned',
       'incident_containment', 'incident_control',
       'incident_cooperating_agencies', 'incident_longitude',
       'incident_latitude', 'incident_type', 'incident_id', 'incident_url',
       'incident_date_extinguished', 'incident_dateonly_extinguished',
       'incident_dateonly_created', 'is_active', 'calfire_incident',
       'notification_desired', 'lon', 'lat', 'fire_start_date', 'rmax_mean_8w',
       'rmin_mean_8w', 'sph_mean_8w', 'srad_mean_8w', 'tmmn_mean_8w',
       'tmmx_mean_8w', 'vs_mean_8w', 'bi_mean_8w', 'fm100_mean_8w',
       'fm1000_mean_8w', 'erc_mean_8w', 'etr_mean_8w', 'pet_mean_8w',
       'vpd_mean_8w', 'pr_sum_8w', 'th_sin_8w', 'th_cos_8w', 'rmax_mean_6w',
       'rmin_mean_6w', 'sph_mean_6w', '

In [12]:
weather_columns = ['incident_id', 'rmax_mean_8w',
       'rmin_mean_8w', 'sph_mean_8w', 'srad_mean_8w', 'tmmn_mean_8w',
       'tmmx_mean_8w', 'vs_mean_8w', 'bi_mean_8w', 'fm100_mean_8w',
       'fm1000_mean_8w', 'erc_mean_8w', 'etr_mean_8w', 'pet_mean_8w',
       'vpd_mean_8w', 'pr_sum_8w', 'th_sin_8w', 'th_cos_8w', 'rmax_mean_6w',
       'rmin_mean_6w', 'sph_mean_6w', 'srad_mean_6w', 'tmmn_mean_6w',
       'tmmx_mean_6w', 'vs_mean_6w', 'bi_mean_6w', 'fm100_mean_6w',
       'fm1000_mean_6w', 'erc_mean_6w', 'etr_mean_6w', 'pet_mean_6w',
       'vpd_mean_6w', 'pr_sum_6w', 'th_sin_6w', 'th_cos_6w', 'rmax_mean_4w',
       'rmin_mean_4w', 'sph_mean_4w', 'srad_mean_4w', 'tmmn_mean_4w',
       'tmmx_mean_4w', 'vs_mean_4w', 'bi_mean_4w', 'fm100_mean_4w',
       'fm1000_mean_4w', 'erc_mean_4w', 'etr_mean_4w', 'pet_mean_4w',
       'vpd_mean_4w', 'pr_sum_4w', 'th_sin_4w', 'th_cos_4w', 'rmax_mean_2w',
       'rmin_mean_2w', 'sph_mean_2w', 'srad_mean_2w', 'tmmn_mean_2w',
       'tmmx_mean_2w', 'vs_mean_2w', 'bi_mean_2w', 'fm100_mean_2w',
       'fm1000_mean_2w', 'erc_mean_2w', 'etr_mean_2w', 'pet_mean_2w',
       'vpd_mean_2w', 'pr_sum_2w', 'th_sin_2w', 'th_cos_2w']

In [13]:
df_weather_selected = df_weather[weather_columns].copy()

In [14]:
df_weather_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3509 entries, 0 to 3508
Data columns (total 69 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   incident_id     3509 non-null   object 
 1   rmax_mean_8w    3484 non-null   float64
 2   rmin_mean_8w    3484 non-null   float64
 3   sph_mean_8w     3484 non-null   float64
 4   srad_mean_8w    3484 non-null   float64
 5   tmmn_mean_8w    3484 non-null   float64
 6   tmmx_mean_8w    3484 non-null   float64
 7   vs_mean_8w      3484 non-null   float64
 8   bi_mean_8w      3484 non-null   float64
 9   fm100_mean_8w   3484 non-null   float64
 10  fm1000_mean_8w  3484 non-null   float64
 11  erc_mean_8w     3484 non-null   float64
 12  etr_mean_8w     3484 non-null   float64
 13  pet_mean_8w     3484 non-null   float64
 14  vpd_mean_8w     3484 non-null   float64
 15  pr_sum_8w       3484 non-null   float64
 16  th_sin_8w       3484 non-null   float64
 17  th_cos_8w       3484 non-null   f

### Vegetation dataset

In [15]:
df_vegetation.columns

Index(['incident_name', 'incident_is_final', 'incident_date_last_update',
       'incident_date_created', 'incident_administrative_unit',
       'incident_administrative_unit_url', 'incident_county',
       'incident_location', 'incident_acres_burned', 'incident_containment',
       'incident_control', 'incident_cooperating_agencies', 'fire_LON',
       'fire_LAT', 'incident_type', 'incident_id', 'incident_url',
       'incident_date_extinguished', 'incident_dateonly_extinguished',
       'incident_dateonly_created', 'is_active', 'calfire_incident',
       'notification_desired', 'fire_date', 'fire_id', 'matched_veg_LAT',
       'matched_veg_LON', 'location_distance_degrees', 'STID', 'DATE',
       'FUEL_TYPE', 'FUEL_VARIATION', 'PERCENT', 'NAME', 'GACC', 'GROUP',
       'veg_LAT', 'veg_LON', 'ELE', 'veg_date', 'date_gap_days'],
      dtype='object')

In [16]:
vegetation_columns = ['incident_id','FUEL_TYPE', 'FUEL_VARIATION', 'PERCENT', 'ELE']

In [17]:
# Select the vegetation columns
df_vegetation_selected = df_vegetation[vegetation_columns].copy()

# Fill the Fuel variation unknown
df_vegetation_selected['FUEL_VARIATION'] = df_vegetation_selected['FUEL_VARIATION'].fillna('Unknown')

In [18]:
df_vegetation_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2655 entries, 0 to 2654
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   incident_id     2655 non-null   object 
 1   FUEL_TYPE       2655 non-null   object 
 2   FUEL_VARIATION  2655 non-null   object 
 3   PERCENT         2655 non-null   float64
 4   ELE             2655 non-null   float64
dtypes: float64(2), object(3)
memory usage: 103.8+ KB


### Topography

In [19]:
df_topography.columns

Index(['incident_name', 'incident_is_final', 'incident_date_last_update',
       'incident_date_created', 'incident_administrative_unit',
       'incident_administrative_unit_url', 'incident_county',
       'incident_location', 'incident_acres_burned', 'incident_containment',
       'incident_control', 'incident_cooperating_agencies',
       'incident_longitude', 'incident_latitude', 'incident_type',
       'incident_id', 'incident_url', 'incident_date_extinguished',
       'incident_dateonly_extinguished', 'incident_dateonly_created',
       'is_active', 'calfire_incident', 'notification_desired', 'created',
       'log_acres', 'aspect', 'elevation', 'first', 'hillshade', 'slope'],
      dtype='object')

In [20]:
topography_columns = ['incident_id', 'aspect', 'elevation', 'first', 'hillshade', 'slope']

In [21]:
df_topography_selected = df_topography[topography_columns].copy()

In [22]:
df_topography_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3405 entries, 0 to 3404
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   incident_id  3405 non-null   object 
 1   aspect       3404 non-null   float64
 2   elevation    3404 non-null   float64
 3   first        3404 non-null   float64
 4   hillshade    3404 non-null   float64
 5   slope        3404 non-null   float64
dtypes: float64(5), object(1)
memory usage: 159.7+ KB


### Drought dataset

In [23]:
df_drought.columns

Index(['incident_name', 'incident_id', 'DSCI_current', 'DSCI_4wk_avg',
       'DSCI_8wk_avg', 'DSCI_12wk_avg', 'DSCI_16wk_avg', 'DSCI_25wk_avg',
       'DSCI_36wk_avg', 'DSCI_42wk_avg', 'DSCI_52wk_avg', 'eddi90d_current',
       'spei180d_current', 'pdsi_current', 'eddi90d_1wk_avg',
       'spei180d_1wk_avg', 'pdsi_1wk_avg', 'eddi90d_2wk_avg',
       'spei180d_2wk_avg', 'pdsi_2wk_avg', 'eddi90d_4wk_avg',
       'spei180d_4wk_avg', 'pdsi_4wk_avg'],
      dtype='object')

In [24]:
drought_columns = ['incident_id', 'DSCI_current', 'DSCI_4wk_avg',
       'DSCI_8wk_avg', 'DSCI_12wk_avg', 'DSCI_16wk_avg', 'DSCI_25wk_avg',
       'DSCI_36wk_avg', 'DSCI_42wk_avg', 'DSCI_52wk_avg', 'eddi90d_current',
       'spei180d_current', 'pdsi_current', 'eddi90d_1wk_avg',
       'spei180d_1wk_avg', 'pdsi_1wk_avg', 'eddi90d_2wk_avg',
       'spei180d_2wk_avg', 'pdsi_2wk_avg', 'eddi90d_4wk_avg',
       'spei180d_4wk_avg', 'pdsi_4wk_avg']

In [25]:
# select the drought columns 
df_drought_selected = df_drought[drought_columns].copy()

# Impute the missing drought columns with mean value
impute_drought_columns = df_drought_selected.columns[10:22]
df_drought_selected = df_drought_selected[drought_columns].fillna(df_drought_selected[impute_drought_columns].median())

In [26]:
df_drought_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3499 entries, 0 to 3498
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   incident_id       3499 non-null   object 
 1   DSCI_current      3492 non-null   float64
 2   DSCI_4wk_avg      3492 non-null   float64
 3   DSCI_8wk_avg      3492 non-null   float64
 4   DSCI_12wk_avg     3492 non-null   float64
 5   DSCI_16wk_avg     3492 non-null   float64
 6   DSCI_25wk_avg     3492 non-null   float64
 7   DSCI_36wk_avg     3492 non-null   float64
 8   DSCI_42wk_avg     3492 non-null   float64
 9   DSCI_52wk_avg     3492 non-null   float64
 10  eddi90d_current   3499 non-null   float64
 11  spei180d_current  3499 non-null   float64
 12  pdsi_current      3499 non-null   float64
 13  eddi90d_1wk_avg   3499 non-null   float64
 14  spei180d_1wk_avg  3499 non-null   float64
 15  pdsi_1wk_avg      3499 non-null   float64
 16  eddi90d_2wk_avg   3499 non-null   float64


### Spatial dataset

In [27]:
df_spatial.columns

Index(['Unnamed: 0', 'incident_id', 'incident_name', 'incident_latitude',
       'incident_longitude', 'incident_county', 'distance_to_road_km',
       'distance_to_city_km', 'distance_to_coast_km',
       'distance_to_powerline_km', 'local_urban_population'],
      dtype='object')

In [28]:
spatial_columns = ['incident_id','distance_to_road_km','distance_to_city_km', 'distance_to_coast_km',
                   'distance_to_powerline_km', 'local_urban_population']

In [29]:
df_spatial_selected = df_spatial[spatial_columns].copy()

In [30]:
df_spatial_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3492 entries, 0 to 3491
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   incident_id               3492 non-null   object 
 1   distance_to_road_km       3492 non-null   float64
 2   distance_to_city_km       3492 non-null   float64
 3   distance_to_coast_km      3492 non-null   float64
 4   distance_to_powerline_km  3492 non-null   float64
 5   local_urban_population    3492 non-null   float64
dtypes: float64(5), object(1)
memory usage: 163.8+ KB


### Rename Columns

In [31]:
df_fire_selected.columns = [
    f"fire_{col.lower()}" for col in df_fire_selected.columns
]

df_weather_selected.columns = [
    f"weather_{col.lower()}" for col in df_weather_selected.columns
]

df_vegetation_selected.columns = [
    f"vegetation_{col.lower()}" for col in df_vegetation_selected.columns
]

df_topography_selected.columns = [
    f"topography_{col.lower()}" for col in df_topography_selected.columns
]

df_drought_selected.columns = [
    f"drought_{col.lower()}" for col in df_drought_selected.columns
]

df_spatial_selected.columns = [
    f"spatial_{col.lower()}" for col in df_spatial_selected.columns
]

In [32]:
df_selected_columns = pd.DataFrame({
    "Dataset": [
        "fire",
        "weather",
        "vegetation",
        "topography",
        "drought",
        "spatial"
    ],
    "Columns": [
        "\n".join(df_fire_selected.columns),
        "\n".join(df_weather_selected.columns),
        "\n".join(df_vegetation_selected.columns),
        "\n".join(df_topography_selected.columns),
        "\n".join(df_drought_selected.columns),
        "\n".join(df_spatial_selected.columns)
    ]
})

In [33]:
display(df_selected_columns.style.set_properties(
    subset=['Columns'],
    **{'white-space': 'pre-wrap'}
))

,Dataset,Columns
0,fire,fire_incident_id fire_incident_county fire_incident_acres_burned
1,weather,weather_incident_id weather_rmax_mean_8w weather_rmin_mean_8w weather_sph_mean_8w weather_srad_mean_8w weather_tmmn_mean_8w weather_tmmx_mean_8w weather_vs_mean_8w weather_bi_mean_8w weather_fm100_mean_8w weather_fm1000_mean_8w weather_erc_mean_8w weather_etr_mean_8w weather_pet_mean_8w weather_vpd_mean_8w weather_pr_sum_8w weather_th_sin_8w weather_th_cos_8w weather_rmax_mean_6w weather_rmin_mean_6w weather_sph_mean_6w weather_srad_mean_6w weather_tmmn_mean_6w weather_tmmx_mean_6w weather_vs_mean_6w weather_bi_mean_6w weather_fm100_mean_6w weather_fm1000_mean_6w weather_erc_mean_6w weather_etr_mean_6w weather_pet_mean_6w weather_vpd_mean_6w weather_pr_sum_6w weather_th_sin_6w weather_th_cos_6w weather_rmax_mean_4w weather_rmin_mean_4w weather_sph_mean_4w weather_srad_mean_4w weather_tmmn_mean_4w weather_tmmx_mean_4w weather_vs_mean_4w weather_bi_mean_4w weather_fm100_mean_4w weather_fm1000_mean_4w weather_erc_mean_4w weather_etr_mean_4w weather_pet_mean_4w weather_vpd_mean_4w weather_pr_sum_4w weather_th_sin_4w weather_th_cos_4w weather_rmax_mean_2w weather_rmin_mean_2w weather_sph_mean_2w weather_srad_mean_2w weather_tmmn_mean_2w weather_tmmx_mean_2w weather_vs_mean_2w weather_bi_mean_2w weather_fm100_mean_2w weather_fm1000_mean_2w weather_erc_mean_2w weather_etr_mean_2w weather_pet_mean_2w weather_vpd_mean_2w weather_pr_sum_2w weather_th_sin_2w weather_th_cos_2w
2,vegetation,vegetation_incident_id vegetation_fuel_type vegetation_fuel_variation vegetation_percent vegetation_ele
3,topography,topography_incident_id topography_aspect topography_elevation topography_first topography_hillshade topography_slope
4,drought,drought_incident_id drought_dsci_current drought_dsci_4wk_avg drought_dsci_8wk_avg drought_dsci_12wk_avg drought_dsci_16wk_avg drought_dsci_25wk_avg drought_dsci_36wk_avg drought_dsci_42wk_avg drought_dsci_52wk_avg drought_eddi90d_current drought_spei180d_current drought_pdsi_current drought_eddi90d_1wk_avg drought_spei180d_1wk_avg drought_pdsi_1wk_avg drought_eddi90d_2wk_avg drought_spei180d_2wk_avg drought_pdsi_2wk_avg drought_eddi90d_4wk_avg drought_spei180d_4wk_avg drought_pdsi_4wk_avg
5,spatial,spatial_incident_id spatial_distance_to_road_km spatial_distance_to_city_km spatial_distance_to_coast_km spatial_distance_to_powerline_km spatial_local_urban_population


In [34]:
# Join Fire and Weather
df_fire_weather_joined = df_fire_selected.merge(df_weather_selected,left_on="fire_incident_id",right_on="weather_incident_id",how="left")
df_fire_weather_joined.drop(columns='weather_incident_id',inplace=True)

# Join Fire, Weather, and Vegetation
df_fire_weather_vegetation_joined = df_fire_weather_joined.merge(df_vegetation_selected,left_on="fire_incident_id",right_on="vegetation_incident_id",how="left")
df_fire_weather_vegetation_joined.drop(columns='vegetation_incident_id',inplace=True)

# Join Fire, Weather, Vegetation, and Topography
df_fire_weather_vegetation_topography_joined = df_fire_weather_vegetation_joined.merge(df_topography_selected,left_on="fire_incident_id",right_on="topography_incident_id",how="left")
df_fire_weather_vegetation_topography_joined.drop(columns='topography_incident_id',inplace=True)

# Join Fire, Weather, Vegetation, Topography, and Drought
df_fire_weather_vegetation_topography_drought_joined = df_fire_weather_vegetation_topography_joined.merge(df_drought_selected,left_on="fire_incident_id",right_on="drought_incident_id",how="left")
df_fire_weather_vegetation_topography_drought_joined.drop(columns='drought_incident_id',inplace=True)

# Join Fire, Weather, Vegetation, Topography, Drought, and Spatial
df_fire_weather_vegetation_topography_drought_spatial_joined = df_fire_weather_vegetation_topography_drought_joined.merge(df_spatial_selected,left_on="fire_incident_id",right_on="spatial_incident_id",how="left")
df_fire_weather_vegetation_topography_drought_spatial_joined.drop(columns='spatial_incident_id',inplace=True)

In [35]:
df_all_joined = df_fire_weather_vegetation_topography_drought_spatial_joined.copy()

In [36]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df_all_joined.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3439 entries, 0 to 3438
Data columns (total 106 columns):
 #    Column                            Non-Null Count  Dtype  
---   ------                            --------------  -----  
 0    fire_incident_id                  3439 non-null   object 
 1    fire_incident_county              3429 non-null   object 
 2    fire_incident_acres_burned        3439 non-null   float64
 3    weather_rmax_mean_8w              3415 non-null   float64
 4    weather_rmin_mean_8w              3415 non-null   float64
 5    weather_sph_mean_8w               3415 non-null   float64
 6    weather_srad_mean_8w              3415 non-null   float64
 7    weather_tmmn_mean_8w              3415 non-null   float64
 8    weather_tmmx_mean_8w              3415 non-null   float64
 9    weather_vs_mean_8w                3415 non-null   float64
 10   weather_bi_mean_8w                3415 non-null   float64
 11   weather_fm100_mean_8w             3415 non-null   floa

In [37]:
df_all_joined.to_csv('../data/processed/calfire_all_ds_joined.csv',index=False)